# EDW REST client (`RESTesri.edw`) — usage examples

Walks through every public function in `RESTesri/edw.py` against the live USFS EDW ArcGIS REST services:

- `search_edw_services` — find services by keyword/theme
- `get_service_info` — service-level metadata (layers, spatial ref)
- `get_layer_info` — layer-level metadata (fields, capabilities)
- `get_layer_metadata` — FGDC/ISO metadata (abstract, field definitions, domains)
- `query_features` — attribute + spatial feature queries
- `query_features_with_pagination` — queries beyond the 2000-record server cap
- `query_features_analytic` — SQL window functions (RANK, SUM, LAG, ...)
- `top_n_per_group` — convenience wrapper for "top N per group" queries

In [3]:
import sys
from pathlib import Path

# This notebook lives in RESTesri/examples/ — add the project root (two levels up)
# to sys.path so `RESTesri` is importable.
sys.path.insert(0, str(Path.cwd().parent.parent))

from RESTesri.edw import (
    search_edw_services,
    get_service_info,
    get_layer_info,
    get_layer_metadata,
    query_features,
    query_features_with_pagination,
    query_features_analytic,
    top_n_per_group,
)

## 1. `search_edw_services` — find services by keyword or theme

Matches on service name, theme description, and a keyword-alias table (e.g. "riparian" pulls in inland-waters/hydro services even though the word never appears in a service name).

In [4]:
# Plain keyword search
fire_services = search_edw_services("fire")
for s in fire_services[:5]:
    print(s["name"], "-", s["theme"])

EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01 - inland_waters
EDW_AerialFireRetardantAvoidanceAreas_Terrestrial_01 - environment
EDW_BurnedAreaEmergencyResponse_01 - environment
EDW_CommWildfireDefenseGrant_01 - environment
EDW_FireOccurrence6thEdition_01 - environment


In [5]:
# Keyword-alias expansion: "riparian" isn't in any service name, but resolves
# to inland_waters/hydro/watershed/aquatic services via _KEYWORD_ALIASES
search_edw_services("riparian")

[{'name': 'EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01/MapServer',
  'theme': 'inland_waters',
  'description': 'This data depicts aquatic aerial fire retardant avoidance areas delivered as part of the 2011 Nationwide Aerial Application of Fire Retardant on…'},
 {'name': 'EDW_AquaticOrganismPassage_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_AquaticOrganismPassage_01/MapServer',
  'theme': 'environment',
  'description': 'This dataset provides USFS watershed improvement activities to barriers to upstream migration.'},
 {'name': 'EDW_ExperimentalForestandRange_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_ExperimentalForestandRange_01/MapServer',
  'theme': 'boundaries',
  'description': 'This polygon feature class contains the boundaries of 86 of 87 experimental 

In [6]:
# Filter by theme only (no keyword) — every transportation-themed service
search_edw_services(theme="transportation")

[{'name': 'EDW_MVUM_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_MVUM_01/MapServer',
  'theme': 'transportation',
  'description': 'The feature class indicates the specific types of motorized vehicles allowed on the designated routes and their seasons of use.'},
 {'name': 'EDW_MVUM_02',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_MVUM_02/MapServer',
  'theme': 'transportation',
  'description': 'A map service on the www depicting Forest Service roads and trails that are designated for motor vehicle use under the official U.S.'},
 {'name': 'EDW_RoadBasic_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_RoadBasic_01/MapServer',
  'theme': 'transportation',
  'description': 'Existing Forest Service roads with attributes representing their characteristics.'},
 {'name': 'EDW_TrailNFSPublish_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/ser

## 2. `get_service_info` — service-level metadata

Returns description, spatial reference, extent, and the list of layers in a MapServer.

In [7]:
service_name = "EDW_MTBS_01"
info = get_service_info(service_name)

print("Service description (truncated at 200 characters):")
print(info["description"][:200])
print("# of layers in service:", len(info["layers"]))

print("Info on first 5 layers:")
for lyr in info["layers"][:5]:
    print(lyr["id"], lyr["name"])

Service description (truncated at 200 characters):
A map service on the www that depicts Fire Occurrence Locations and Burned Area Boundaries from the beginning of the Landsat Thematic Mapper archive to the present. The Monitoring Trends in Burn Sever
# of layers in service: 84
Info on first 5 layers:
83 2024 Fire Occurrence Locations
82 2024 Burned Area Boundaries
81 2023 Fire Occurrence Locations
80 2023 Burned Area Boundaries
78 2022 Fire Occurrence Locations


## 3. `get_layer_info` — layer-level metadata

Fields, geometry type, and which operations/analytics the layer supports (`capabilities`, `advancedQueryCapabilities`).

In [8]:
layer_id = 63  # "Burned Area Boundaries (All Years)"
layer_info = get_layer_info(service_name, layer_id)
print(layer_info["name"], layer_info["geometryType"])
print("capabilities:", layer_info["capabilities"])
[f["name"] for f in layer_info["fields"]][:10]

Burned Area Boundaries (All Years) esriGeometryPolygon
capabilities: ['Map', 'Query', 'Data']


['objectid',
 'fire_id',
 'fire_name',
 'year',
 'startmonth',
 'startday',
 'fire_type',
 'acres',
 'irwinid',
 'map_id']

## 4. `get_layer_metadata` — FGDC/ISO metadata document

A separate document from `get_layer_info` — carries the dataset abstract/purpose and per-field *definitions* (what a field actually means), plus optional coded-value/range domains.

In [12]:
meta = get_layer_metadata(service_name, layer_id, include_domains=True)
print(meta["title"])
print(meta["abstract"][:300])
print("keywords:", meta["keywords"][:5])

# Field definitions (only some fields are documented by the data provider)
for attr in meta["attributes"]:
    print(attr["name"], "->", attr["definition"] or "(undocumented)")

MTBS_Burn_Area_Boundary
The Monitoring Trends in Burn Severity (MTBS) Program assesses the frequency, extent, and magnitude (size and severity) of all large wildland fires (including wildfires and prescribed fires) in the conterminous United States (CONUS), Alaska, Hawaii, and Puerto Rico from the beginning of the Landsat 
keywords: ['Burn severity', 'Burned area', 'Differenced normalized burn ratio', 'Fire location', 'Fire occurrence']
OBJECTID -> Internal feature number.
FIRE_ID -> (undocumented)
FIRE_NAME -> (undocumented)
YEAR -> (undocumented)
STARTMONTH -> (undocumented)
STARTDAY -> (undocumented)
FIRE_TYPE -> (undocumented)
ACRES -> (undocumented)
Post_ID -> Landsat or Sentinel post scene ID.
Pre_ID -> Landsat or Sentinel pre scene ID.
Asmnt_Type -> Fire mapping assessment label (Initial (SS) (SS=single scene), Initial, Extended, Extended (SS) (SS=single scene), Emergency, or Emergency (SS)).
Shape -> Feature geometry.
dNBR_offst -> The mean dNBR value sampled from an unburned a

## 5. `query_features` — attribute and spatial queries

Returns a GeoJSON `FeatureCollection`. `where` filters by attributes; `geometry`/`geometry_type`/`spatial_rel` add a spatial filter.

In [10]:
# Attribute-only query: a single named fire
cameron_peak = query_features(
    service_name, layer_id,
    where="fire_name = 'CAMERON PEAK'",
    out_fields="fire_name,ig_date,acres",
)
cameron_peak["features"]

KeyboardInterrupt: 

In [ ]:
# return_count_only avoids pulling geometry/attributes when you only need a number
query_features(service_name, layer_id, where="acres > 100000", return_count_only=True)

{'count': 308}

In [23]:
# Spatial query: use a forest boundary as the AOI to clip another layer.
# Grab the Idaho Panhandle NF boundary — a 161-part multipolygon with holes.
# query_features automatically falls back to an ID-based fetch for AOIs this
# geometrically complex (see the Note in its docstring), so this just works.
ipnf = query_features(
    "EDW_ForestSystemBoundaries_01", 0,
    where="FORESTORGCODE='0104'",
)
aoi_geom = ipnf["features"][0]["geometry"]

# ...then query fires intersecting it (geometry can be a GeoJSON dict, Esri JSON, or a bbox string)
ipnf_fires = query_features(
    service_name, layer_id,
    geometry=aoi_geom, geometry_type="esriGeometryPolygon",
    out_fields="fire_name,year,acres",
)
print(len(ipnf_fires["features"]), "fires intersecting the IPNF boundary")
for p in sorted((f["properties"] for f in ipnf_fires["features"]), key=lambda p: p["year"])[-5:]:
    print(f"{p['fire_name']} ({p['year']}): {p['acres']:,.0f} acres")

40 fires intersecting the IPNF boundary
CALEDONIA (2022): 1,451 acres
RUSSELL MOUNTAIN (2022): 26,104 acres
DIAMOND WATCH (2022): 1,432 acres
THOR (2022): 2,097 acres
RIDGE CREEK (2023): 4,600 acres


In [ ]:
# A plain bbox string also works as the geometry filter
bbox_fires = query_features(
    service_name, layer_id,
    geometry="-116.5,47.5,-116.0,48.0", geometry_type="esriGeometryEnvelope",
    out_fields="fire_name,year,acres", max_features=5,
)
for f in bbox_fires["features"]:
    p = f["properties"]
    print(f"{p['fire_name']} ({p['year']}): {p['acres']:,.0f} acres")

ULM PEAK (2006): 4,606 acres
CAPE HORN (2015): 1,505 acres
NORTH GRIZZLY (2015): 5,284 acres
WHITETAIL (2015): 2,022 acres
LOWER FLAT (2015): 9,023 acres


## 6. `query_features_with_pagination` — beyond the 2000-record server cap

Same signature as `query_features`, but `max_features` can exceed `_MAX_RECORD_COUNT`; it pages through `resultOffset` automatically. Using the point layer (fire ignition locations) here rather than the polygon layer — the EDW server 500s on exactly-2000-row pages of heavy polygon geometry, a server-side quirk unrelated to this function.

In [ ]:
all_fires = query_features_with_pagination(
    service_name, 62,  # "Fire Occurrence Locations (All Years)" — points, not polygons
    out_fields="fire_name,ig_date,acres",
    max_features=4000,
)
len(all_fires["features"])

4000

## 7. `query_features_analytic` — SQL window functions

Runs ArcGIS's `queryAnalytic` operation (RANK, SUM, LAG/LEAD, PERCENTILE_CONT, ...). Unlike `query_features`, rows aren't collapsed — each analytic value is appended as a new field on its source feature.

Note: ArcGIS ignores whatever `out_name` you request for a `RANK` analytic and always names the computed field `rank_expr0` — filter on that name in `analytic_where`, not your requested `out_name`.

In [ ]:
# Rank fires by acreage within each ignition year, keep only the #1 fire per year
biggest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{
        "type": "RANK",
        "field": "acres",
        "order_by": "acres DESC",
        "out_name": "acres_rank",
    }],
    partition_by="year",
    analytic_where="rank_expr0 = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
sorted(
    (f["properties"] for f in biggest_per_year["features"]),
    key=lambda p: p["year"],
)[-5:]

[{'fire_name': 'HERMITS PEAK',
  'year': 2022,
  'acres': 351782.0,
  'rank_expr0': 1},
 {'fire_name': 'YORK', 'year': 2023, 'acres': 94728.0, 'rank_expr0': 1},
 {'fire_name': 'SMOKEHOUSE CREEK',
  'year': 2024,
  'acres': 1047245.0,
  'rank_expr0': 1},
 {'fire_name': 'COTTONWOOD PEAK',
  'year': 2025,
  'acres': 137796.0,
  'rank_expr0': 1},
 {'fire_name': 'MORRILL', 'year': 2026, 'acres': 645316.0, 'rank_expr0': 1}]

**Other analytic types:** the Esri `queryAnalytic` spec also defines `SUM`, `AVG`, `MIN`, `MAX`, `COUNT`, `STDDEV`, `VAR`, `LAG`, `LEAD`, `NTILE`, `FIRST_VALUE`, `LAST_VALUE`, `PERCENTILE_CONT`, and `PERCENTILE_DISC` — but this EDW server only supports the ranking-family functions in practice: `RANK` (above), `DENSE_RANK`, `ROW_NUMBER`, `PERCENT_RANK`, and `CUME_DIST`. The rest fail with `"Unable to complete operation"` (or an explicit `"not supported"` for the percentile functions), regardless of how they're called.

`ROW_NUMBER` is a useful contrast to `RANK`: `RANK` gives tied values the *same* rank, so a tie for smallest/largest fire in a given year produces multiple rows for that year (e.g. 2024 has two fires tied at exactly 501 acres). `ROW_NUMBER` breaks ties arbitrarily but deterministically, guaranteeing exactly one row per partition — useful when you need a strict 1-per-group result regardless of ties.

In [11]:
# Smallest fire per year via ROW_NUMBER — exactly one row per year, even in years
# (like 2024) where two fires are tied at exactly the same acreage
smallest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{
        "type": "ROW_NUMBER",
        "order_by": "acres ASC",
        "out_name": "row_num",
    }],
    partition_by="year",
    analytic_where="row_number_expr0 = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
print(len(smallest_per_year["features"]), "features (one per year)")
for f in sorted((f["properties"] for f in smallest_per_year["features"]), key=lambda p: p["year"])[-5:]:
    print(f)

43 features (one per year)
{'fire_name': 'C-136 QUAIL 26/32-137B QUAIL 28 FM', 'year': 2022, 'acres': 502.0, 'row_number_expr0': 1}
{'fire_name': 'ROUND MOUNTAIN', 'year': 2023, 'acres': 500.0, 'row_number_expr0': 1}
{'fire_name': 'C-212 BB1 LC RX', 'year': 2024, 'acres': 501.0, 'row_number_expr0': 1}
{'fire_name': 'CAHAS MOUNTTAIN FIRE', 'year': 2025, 'acres': 501.0, 'row_number_expr0': 1}
{'fire_name': 'SALINA CANYON EAST RX', 'year': 2026, 'acres': 5058.0, 'row_number_expr0': 1}


## 8. `top_n_per_group` — convenience wrapper

Same query as above, without hand-rolling the RANK analytic. `descending=False` ranks smallest-first instead.

In [50]:
biggest_by_year = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=True,
    n=1,
    out_fields="fire_name,year,acres",
)

sorted(
    (f["properties"] for f in biggest_by_year["features"]),
    key=lambda p: p["year"],
)[-5:]

[{'fire_name': 'HERMITS PEAK',
  'year': 2022,
  'acres': 351782.0,
  'rank_expr0': 1},
 {'fire_name': 'YORK', 'year': 2023, 'acres': 94728.0, 'rank_expr0': 1},
 {'fire_name': 'SMOKEHOUSE CREEK',
  'year': 2024,
  'acres': 1047245.0,
  'rank_expr0': 1},
 {'fire_name': 'COTTONWOOD PEAK',
  'year': 2025,
  'acres': 137796.0,
  'rank_expr0': 1},
 {'fire_name': 'MORRILL', 'year': 2026, 'acres': 645316.0, 'rank_expr0': 1}]

Smallest fire by year. Note this function will silently return more than N entries per group if there is a tie. (See 2024 duplicate below). If you need a single entry per group, use query_analytic() with ROW_NUMBER.

In [21]:
smallest_by_year = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=False, # set to false for smallest first
    n=1,
    out_fields="fire_name,year,acres",
)

sorted(
    (f["properties"] for f in smallest_by_year["features"]),
    key=lambda p: p["year"],
)[-5:]

[{'fire_name': 'BOGGY HOLLOW SOUTH',
  'year': 2023,
  'acres': 500.0,
  'rank_expr0': 1},
 {'fire_name': 'C-212 BB1 LC RX',
  'year': 2024,
  'acres': 501.0,
  'rank_expr0': 1},
 {'fire_name': 'CON RKC 01 RX', 'year': 2024, 'acres': 501.0, 'rank_expr0': 1},
 {'fire_name': 'CAHAS MOUNTTAIN FIRE',
  'year': 2025,
  'acres': 501.0,
  'rank_expr0': 1},
 {'fire_name': 'SALINA CANYON EAST RX',
  'year': 2026,
  'acres': 5058.0,
  'rank_expr0': 1}]

Get the largest fire that started that started in each month across all years.

In [17]:
biggest_by_month = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="startmonth",
    descending=True,
    n=1,
    out_fields="fire_name,year,startmonth,acres",
)

sorted(
    (f["properties"] for f in biggest_by_month["features"]),
    key=lambda p: p["startmonth"],
)

[{'fire_name': 'UNNAMED',
  'year': 2014,
  'startmonth': 1,
  'acres': 60996.0,
  'rank_expr0': 1},
 {'fire_name': 'SMOKEHOUSE CREEK',
  'year': 2024,
  'startmonth': 2,
  'acres': 1047245.0,
  'rank_expr0': 1},
 {'fire_name': 'OKS - STARBUCK',
  'year': 2017,
  'startmonth': 3,
  'acres': 656868.0,
  'rank_expr0': 1},
 {'fire_name': 'HERMITS PEAK',
  'year': 2022,
  'startmonth': 4,
  'acres': 351782.0,
  'rank_expr0': 1},
 {'fire_name': 'WALLOW',
  'year': 2011,
  'startmonth': 5,
  'acres': 563564.0,
  'rank_expr0': 1},
 {'fire_name': 'BOUNDARY',
  'year': 2004,
  'startmonth': 6,
  'acres': 537728.0,
  'rank_expr0': 1},
 {'fire_name': 'DIXIE',
  'year': 2021,
  'startmonth': 7,
  'acres': 979722.0,
  'rank_expr0': 1},
 {'fire_name': 'AUGUST COMPLEX',
  'year': 2020,
  'startmonth': 8,
  'acres': 1067970.0,
  'rank_expr0': 1},
 {'fire_name': 'CREEK',
  'year': 2020,
  'startmonth': 9,
  'acres': 381529.0,
  'rank_expr0': 1},
 {'fire_name': 'CEDAR',
  'year': 2003,
  'startmonth': 1

Look at the 10 largest fires just in 2017. 

In [20]:
largest_2017 = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=True,
    n=10,
    where="year=2017",
    out_fields="fire_name,year,startmonth,acres",
)

sorted(
    (f["properties"] for f in largest_2017["features"]),
    key=lambda p: p["year"],
)

[{'fire_name': 'OKS - STARBUCK',
  'year': 2017,
  'startmonth': 3,
  'acres': 656868.0,
  'rank_expr0': 1},
 {'fire_name': 'PERRYTON',
  'year': 2017,
  'startmonth': 3,
  'acres': 290085.0,
  'rank_expr0': 2},
 {'fire_name': 'THOMAS',
  'year': 2017,
  'startmonth': 12,
  'acres': 282076.0,
  'rank_expr0': 3},
 {'fire_name': 'BRIDGE COULEE',
  'year': 2017,
  'startmonth': 7,
  'acres': 222645.0,
  'rank_expr0': 4},
 {'fire_name': 'ROOSTERS COMB',
  'year': 2017,
  'startmonth': 7,
  'acres': 217340.0,
  'rank_expr0': 5},
 {'fire_name': 'CHETCO BAR',
  'year': 2017,
  'startmonth': 7,
  'acres': 194761.0,
  'rank_expr0': 6},
 {'fire_name': 'RICE RIDGE',
  'year': 2017,
  'startmonth': 7,
  'acres': 171675.0,
  'rank_expr0': 7},
 {'fire_name': 'WEST MIMS',
  'year': 2017,
  'startmonth': 4,
  'acres': 166678.0,
  'rank_expr0': 8},
 {'fire_name': 'CAMPBELL RIVER',
  'year': 2017,
  'startmonth': 6,
  'acres': 161363.0,
  'rank_expr0': 9},
 {'fire_name': 'SNOWSTORM',
  'year': 2017,
  '